# The boundary returns

## A clean core is not a clean system
In [Part I](../part1/01-blub-paradox.ipynb) you watched five rungs of the
language ladder close the seven footguns in the *domain core*. By the time
we landed on Rust, the decision engine was pure, the types made impossible
states unrepresentable, exhaustive `match` caught missing cases at compile
time, and the batch summary was derived from per-row decisions instead of
mutated alongside them. The blub-paradox demo had landed: there was a
class of bug each rung made structurally impossible.

So why is there a Part Ib?

Because real systems do not take pre-validated `ParsedFuelRow` values.
Real systems take CSV strings. They call a vehicle-lookup service that
sometimes times out. They write audit events to a log the ops team
actually reads. They emit an operational report that drives a "what
uploaded today" dashboard. Every one of those is an *edge*, a place
where the typed domain has to meet messy reality — and every one is a
*new* surface where bad types can sneak back in.

V2 of this experiment asked: *can the language model the domain
cleanly?* V3 asked a harder question: *can the language keep that
domain clean when integration arrives?*

That is what Part Ib is about.

## What V3 added

The V3 pass kept the V2 domain rules frozen and added four boundary
concerns around the pure core:

1. **CSV-shaped import.** Already-parsed CSV cells (every field a
   nullable string) flow in. The mapper turns them into typed application
   DTOs or typed import errors. Nothing about being a CSV cell ever
   crosses into the decision engine.
2. **Repository ports.** Vehicle lookup and duplicate history live
   behind application-layer interfaces. The pure engine sees typed
   `VehicleLookupResult` and `DuplicateCheckResult` values, never a
   `SqlConnection` or an HTTP client.
3. **Audit projection.** A separate projector reads `BatchDecision` and
   emits typed audit records, with a clearly-marked string conversion at
   the very edge where they leave the process.
4. **Operational batch report.** A second projector reads the same
   `BatchDecision` and emits the ops-facing report: counts, uploaded
   transaction IDs, rejected row numbers, quarantined rows with reasons,
   and a fatal/ready status.

The full V3 pipeline looks like this:

<div class="mermaid-diagram" style="width:100%">
<svg id="mermaid-figure-1" xmlns="http://www.w3.org/2000/svg" xlink="http://www.w3.org/1999/xlink" class="flowchart" viewbox="0 0 2375.875 382" role="graphics-document document" aria-roledescription="flowchart-v2" style="width:100%;height:auto;max-width:100%;display:block">
<style>#mermaid-figure-1{font-family:"trebuchet ms",verdana,arial,sans-serif;font-size:16px;fill:#000000;}@keyframes edge-animation-frame{from{stroke-dashoffset:0;}}@keyframes dash{to{stroke-dashoffset:0;}}#mermaid-figure-1 .edge-animation-slow{stroke-dasharray:9,5!important;stroke-dashoffset:900;animation:dash 50s linear infinite;stroke-linecap:round;}#mermaid-figure-1 .edge-animation-fast{stroke-dasharray:9,5!important;stroke-dashoffset:900;animation:dash 20s linear infinite;stroke-linecap:round;}#mermaid-figure-1 .error-icon{fill:#552222;}#mermaid-figure-1 .error-text{fill:#552222;stroke:#552222;}#mermaid-figure-1 .edge-thickness-normal{stroke-width:1px;}#mermaid-figure-1 .edge-thickness-thick{stroke-width:3.5px;}#mermaid-figure-1 .edge-pattern-solid{stroke-dasharray:0;}#mermaid-figure-1 .edge-thickness-invisible{stroke-width:0;fill:none;}#mermaid-figure-1 .edge-pattern-dashed{stroke-dasharray:3;}#mermaid-figure-1 .edge-pattern-dotted{stroke-dasharray:2;}#mermaid-figure-1 .marker{fill:#666;stroke:#666;}#mermaid-figure-1 .marker.cross{stroke:#666;}#mermaid-figure-1 svg{font-family:"trebuchet ms",verdana,arial,sans-serif;font-size:16px;}#mermaid-figure-1 p{margin:0;}#mermaid-figure-1 .label{font-family:"trebuchet ms",verdana,arial,sans-serif;color:#000000;}#mermaid-figure-1 .cluster-label text{fill:#333;}#mermaid-figure-1 .cluster-label span{color:#333;}#mermaid-figure-1 .cluster-label span p{background-color:transparent;}#mermaid-figure-1 .label text,#mermaid-figure-1 span{fill:#000000;color:#000000;}#mermaid-figure-1 .node rect,#mermaid-figure-1 .node circle,#mermaid-figure-1 .node ellipse,#mermaid-figure-1 .node polygon,#mermaid-figure-1 .node path{fill:#eee;stroke:#999;stroke-width:1px;}#mermaid-figure-1 .rough-node .label text,#mermaid-figure-1 .node .label text,#mermaid-figure-1 .image-shape .label,#mermaid-figure-1 .icon-shape .label{text-anchor:middle;}#mermaid-figure-1 .node .katex path{fill:#000;stroke:#000;stroke-width:1px;}#mermaid-figure-1 .rough-node .label,#mermaid-figure-1 .node .label,#mermaid-figure-1 .image-shape .label,#mermaid-figure-1 .icon-shape .label{text-align:center;}#mermaid-figure-1 .node.clickable{cursor:pointer;}#mermaid-figure-1 .root .anchor path{fill:#666!important;stroke-width:0;stroke:#666;}#mermaid-figure-1 .arrowheadPath{fill:#333333;}#mermaid-figure-1 .edgePath .path{stroke:#666;stroke-width:2.0px;}#mermaid-figure-1 .flowchart-link{stroke:#666;fill:none;}#mermaid-figure-1 .edgeLabel{background-color:white;text-align:center;}#mermaid-figure-1 .edgeLabel p{background-color:white;}#mermaid-figure-1 .edgeLabel rect{opacity:0.5;background-color:white;fill:white;}#mermaid-figure-1 .labelBkg{background-color:rgba(255, 255, 255, 0.5);}#mermaid-figure-1 .cluster rect{fill:hsl(0, 0%, 98.9215686275%);stroke:#707070;stroke-width:1px;}#mermaid-figure-1 .cluster text{fill:#333;}#mermaid-figure-1 .cluster span{color:#333;}#mermaid-figure-1 div.mermaidTooltip{position:absolute;text-align:center;max-width:200px;padding:2px;font-family:"trebuchet ms",verdana,arial,sans-serif;font-size:12px;background:hsl(-160, 0%, 93.3333333333%);border:1px solid #707070;border-radius:2px;pointer-events:none;z-index:100;}#mermaid-figure-1 .flowchartTitleText{text-anchor:middle;font-size:18px;fill:#000000;}#mermaid-figure-1 rect.text{fill:none;stroke-width:0;}#mermaid-figure-1 .icon-shape,#mermaid-figure-1 .image-shape{background-color:white;text-align:center;}#mermaid-figure-1 .icon-shape p,#mermaid-figure-1 .image-shape p{background-color:white;padding:2px;}#mermaid-figure-1 .icon-shape rect,#mermaid-figure-1 .image-shape rect{opacity:0.5;background-color:white;fill:white;}#mermaid-figure-1 .label-icon{display:inline-block;height:1em;overflow:visible;vertical-align:-0.125em;}#mermaid-figure-1 .node .label-icon path{fill:currentColor;stroke:revert;stroke-width:revert;}#mermaid-figure-1 :root{--mermaid-font-family:"trebuchet ms",verdana,arial,sans-serif;}</style>
<g><marker id="mermaid-1779381583394_flowchart-v2-pointEnd" class="marker flowchart-v2" viewbox="0 0 10 10" refx="5" refy="5" markerunits="userSpaceOnUse" markerwidth="8" markerheight="8" orient="auto"><path d="M 0 0 L 10 5 L 0 10 z" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></path></marker><marker id="mermaid-1779381583394_flowchart-v2-pointStart" class="marker flowchart-v2" viewbox="0 0 10 10" refx="4.5" refy="5" markerunits="userSpaceOnUse" markerwidth="8" markerheight="8" orient="auto"><path d="M 0 5 L 10 10 L 10 0 z" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></path></marker><marker id="mermaid-1779381583394_flowchart-v2-circleEnd" class="marker flowchart-v2" viewbox="0 0 10 10" refx="11" refy="5" markerunits="userSpaceOnUse" markerwidth="11" markerheight="11" orient="auto"><circle cx="5" cy="5" r="5" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></circle></marker><marker id="mermaid-1779381583394_flowchart-v2-circleStart" class="marker flowchart-v2" viewbox="0 0 10 10" refx="-1" refy="5" markerunits="userSpaceOnUse" markerwidth="11" markerheight="11" orient="auto"><circle cx="5" cy="5" r="5" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></circle></marker><marker id="mermaid-1779381583394_flowchart-v2-crossEnd" class="marker cross flowchart-v2" viewbox="0 0 11 11" refx="12" refy="5.2" markerunits="userSpaceOnUse" markerwidth="11" markerheight="11" orient="auto"><path d="M 1,1 l 9,9 M 10,1 l -9,9" class="arrowMarkerPath" style="stroke-width: 2; stroke-dasharray: 1, 0;"></path></marker><marker id="mermaid-1779381583394_flowchart-v2-crossStart" class="marker cross flowchart-v2" viewbox="0 0 11 11" refx="-1" refy="5.2" markerunits="userSpaceOnUse" markerwidth="11" markerheight="11" orient="auto"><path d="M 1,1 l 9,9 M 10,1 l -9,9" class="arrowMarkerPath" style="stroke-width: 2; stroke-dasharray: 1, 0;"></path></marker><g class="root"><g class="clusters"></g><g class="edgePaths"><path d="M268,87L272.167,87C276.333,87,284.667,87,292.333,87C300,87,307,87,310.5,87L314,87" id="L_CSV_ImportMapper_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_CSV_ImportMapper_0" data-points="W3sieCI6MjY4LCJ5Ijo4N30seyJ4IjoyOTMsInkiOjg3fSx7IngiOjMxOCwieSI6ODd9XQ==" marker-end="url(#mermaid-1779381583394_flowchart-v2-pointEnd)"></path><path d="M467.369,60L477.762,55.833C488.155,51.667,508.941,43.333,526.614,39.167C544.286,35,558.846,35,566.126,35L573.406,35" id="L_ImportMapper_AppDto_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_ImportMapper_AppDto_0" data-points="W3sieCI6NDY3LjM2OTI5MDg2NTM4NDY0LCJ5Ijo2MH0seyJ4Ijo1MjkuNzI2NTYyNSwieSI6MzV9LHsieCI6NTc3LjQwNjI1LCJ5IjozNX1d" marker-end="url(#mermaid-1779381583394_flowchart-v2-pointEnd)"></path><path d="M467.369,114L477.762,118.167C488.155,122.333,508.941,130.667,529.555,134.833C550.169,139,570.612,139,580.833,139L591.055,139" id="L_ImportMapper_ImportErrors_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_ImportMapper_ImportErrors_0" data-points="W3sieCI6NDY3LjM2OTI5MDg2NTM4NDY0LCJ5IjoxMTR9LHsieCI6NTI5LjcyNjU2MjUsInkiOjEzOX0seyJ4Ijo1OTUuMDU0Njg3NSwieSI6MTM5fV0=" marker-end="url(#mermaid-1779381583394_flowchart-v2-pointEnd)"></path><path d="M811.422,35L815.589,35C819.755,35,828.089,35,853.515,64.625C878.941,94.25,921.46,153.5,942.72,183.125L963.98,212.75" id="L_AppDto_RepoService_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_AppDto_RepoService_0" data-points="W3sieCI6ODExLjQyMTg3NSwieSI6MzV9LHsieCI6ODM2LjQyMTg3NSwieSI6MzV9LHsieCI6OTY2LjMxMTY3MzY3Nzg4NDYsInkiOjIxNn1d" marker-end="url(#mermaid-1779381583394_flowchart-v2-pointEnd)"></path><path d="M803.57,243L809.046,243C814.521,243,825.471,243,834.447,243C843.422,243,850.422,243,853.922,243L857.422,243" id="L_VehiclePort_RepoService_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_VehiclePort_RepoService_0" data-points="W3sieCI6ODAzLjU3MDMxMjUsInkiOjI0M30seyJ4Ijo4MzYuNDIxODc1LCJ5IjoyNDN9LHsieCI6ODYxLjQyMTg3NSwieSI6MjQzfV0=" marker-end="url(#mermaid-1779381583394_flowchart-v2-pointEnd)"></path><path d="M811.125,347L815.341,347C819.557,347,827.99,347,850.078,334.548C872.166,322.096,907.91,297.191,925.782,284.739L943.654,272.287" id="L_DuplicatePort_RepoService_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_DuplicatePort_RepoService_0" data-points="W3sieCI6ODExLjEyNSwieSI6MzQ3fSx7IngiOjgzNi40MjE4NzUsInkiOjM0N30seyJ4Ijo5NDYuOTM1ODQ3MzU1NzY5MywieSI6MjcwfV0=" marker-end="url(#mermaid-1779381583394_flowchart-v2-pointEnd)"></path><path d="M1109.953,243L1114.12,243C1118.286,243,1126.62,243,1134.286,243C1141.953,243,1148.953,243,1152.453,243L1155.953,243" id="L_RepoService_DomainInput_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_RepoService_DomainInput_0" data-points="W3sieCI6MTEwOS45NTMxMjUsInkiOjI0M30seyJ4IjoxMTM0Ljk1MzEyNSwieSI6MjQzfSx7IngiOjExNTkuOTUzMTI1LCJ5IjoyNDN9XQ==" marker-end="url(#mermaid-1779381583394_flowchart-v2-pointEnd)"></path><path d="M1357.797,243L1361.964,243C1366.13,243,1374.464,243,1382.13,243C1389.797,243,1396.797,243,1400.297,243L1403.797,243" id="L_DomainInput_Engine_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_DomainInput_Engine_0" data-points="W3sieCI6MTM1Ny43OTY4NzUsInkiOjI0M30seyJ4IjoxMzgyLjc5Njg3NSwieSI6MjQzfSx7IngiOjE0MDcuNzk2ODc1LCJ5IjoyNDN9XQ==" marker-end="url(#mermaid-1779381583394_flowchart-v2-pointEnd)"></path><path d="M1568.281,243L1572.448,243C1576.615,243,1584.948,243,1592.615,243C1600.281,243,1607.281,243,1610.781,243L1614.281,243" id="L_Engine_Decision_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_Engine_Decision_0" data-points="W3sieCI6MTU2OC4yODEyNSwieSI6MjQzfSx7IngiOjE1OTMuMjgxMjUsInkiOjI0M30seyJ4IjoxNjE4LjI4MTI1LCJ5IjoyNDN9XQ==" marker-end="url(#mermaid-1779381583394_flowchart-v2-pointEnd)"></path><path d="M1754.533,216L1763.038,211.833C1771.543,207.667,1788.553,199.333,1808.265,195.167C1827.977,191,1850.391,191,1861.598,191L1872.805,191" id="L_Decision_AuditProj_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_Decision_AuditProj_0" data-points="W3sieCI6MTc1NC41MzMzNTMzNjUzODQ1LCJ5IjoyMTZ9LHsieCI6MTgwNS41NjI1LCJ5IjoxOTF9LHsieCI6MTg3Ni44MDQ2ODc1LCJ5IjoxOTF9XQ==" marker-end="url(#mermaid-1779381583394_flowchart-v2-pointEnd)"></path><path d="M1754.533,270L1763.038,274.167C1771.543,278.333,1788.553,286.667,1800.558,290.833C1812.563,295,1819.563,295,1823.063,295L1826.563,295" id="L_Decision_ReportProj_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_Decision_ReportProj_0" data-points="W3sieCI6MTc1NC41MzMzNTMzNjUzODQ1LCJ5IjoyNzB9LHsieCI6MTgwNS41NjI1LCJ5IjoyOTV9LHsieCI6MTgzMC41NjI1LCJ5IjoyOTV9XQ==" marker-end="url(#mermaid-1779381583394_flowchart-v2-pointEnd)"></path><path d="M2039.977,191L2051.85,191C2063.724,191,2087.471,191,2105.984,191C2124.497,191,2137.776,191,2144.415,191L2151.055,191" id="L_AuditProj_Audit_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_AuditProj_Audit_0" data-points="W3sieCI6MjAzOS45NzY1NjI1LCJ5IjoxOTF9LHsieCI6MjExMS4yMTg3NSwieSI6MTkxfSx7IngiOjIxNTUuMDU0Njg3NSwieSI6MTkxfV0=" marker-end="url(#mermaid-1779381583394_flowchart-v2-pointEnd)"></path><path d="M2086.219,295L2090.385,295C2094.552,295,2102.885,295,2110.552,295C2118.219,295,2125.219,295,2128.719,295L2132.219,295" id="L_ReportProj_Report_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_ReportProj_Report_0" data-points="W3sieCI6MjA4Ni4yMTg3NSwieSI6Mjk1fSx7IngiOjIxMTEuMjE4NzUsInkiOjI5NX0seyJ4IjoyMTM2LjIxODc1LCJ5IjoyOTV9XQ==" marker-end="url(#mermaid-1779381583394_flowchart-v2-pointEnd)"></path></g><g class="edgeLabels"><g class="edgeLabel"><g class="label" data-id="L_CSV_ImportMapper_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel" transform="translate(529.7265625, 35)"><g class="label" data-id="L_ImportMapper_AppDto_0" transform="translate(-16.453125, -12)"><foreignobject width="32.90625" height="24">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel">
<p>
valid
</p>
</span>
</div>
</foreignobject></g></g><g class="edgeLabel" transform="translate(529.7265625, 139)"><g class="label" data-id="L_ImportMapper_ImportErrors_0" transform="translate(-22.6796875, -12)"><foreignobject width="45.359375" height="24">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel">
<p>
invalid
</p>
</span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_AppDto_RepoService_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_VehiclePort_RepoService_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_DuplicatePort_RepoService_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_RepoService_DomainInput_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_DomainInput_Engine_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_Engine_Decision_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_Decision_AuditProj_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_Decision_ReportProj_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_AuditProj_Audit_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_ReportProj_Report_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g></g><g class="nodes"><g class="node default" id="flowchart-CSV-0" transform="translate(138, 87)"><rect class="basic label-container" style="" x="-130" y="-39" width="260" height="78"></rect><g class="label" style="" transform="translate(-100, -24)"><rect></rect><foreignobject width="200" height="48">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table; white-space: break-spaces; line-height: 1.5; max-width: 200px; text-align: center; width: 200px;">
<span class="nodeLabel">
<p>
CSV rows: strings, nullable cells
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-ImportMapper-1" transform="translate(400.0234375, 87)"><rect class="basic label-container" style="" x="-82.0234375" y="-27" width="164.046875" height="54"></rect><g class="label" style="" transform="translate(-52.0234375, -12)"><rect></rect><foreignobject width="104.046875" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
Import mapper
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-AppDto-3" transform="translate(694.4140625, 35)"><rect class="basic label-container" style="" x="-117.0078125" y="-27" width="234.015625" height="54"></rect><g class="label" style="" transform="translate(-87.0078125, -12)"><rect></rect><foreignobject width="174.015625" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
Application DTO request
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-ImportErrors-5" transform="translate(694.4140625, 139)"><rect class="basic label-container" style="" x="-99.359375" y="-27" width="198.71875" height="54"></rect><g class="label" style="" transform="translate(-69.359375, -12)"><rect></rect><foreignobject width="138.71875" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
Typed import errors
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-RepoService-7" transform="translate(985.6875, 243)"><rect class="basic label-container" style="" x="-124.265625" y="-27" width="248.53125" height="54"></rect><g class="label" style="" transform="translate(-94.265625, -12)"><rect></rect><foreignobject width="188.53125" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
Repository-backed service
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-VehiclePort-8" transform="translate(694.4140625, 243)"><rect class="basic label-container" style="" x="-109.15625" y="-27" width="218.3125" height="54"></rect><g class="label" style="" transform="translate(-79.15625, -12)"><rect></rect><foreignobject width="158.3125" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
Vehicle repository port
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-DuplicatePort-10" transform="translate(694.4140625, 347)"><rect class="basic label-container" style="" x="-116.7109375" y="-27" width="233.421875" height="54"></rect><g class="label" style="" transform="translate(-86.7109375, -12)"><rect></rect><foreignobject width="173.421875" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
Duplicate repository port
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-DomainInput-13" transform="translate(1258.875, 243)"><rect class="basic label-container" style="" x="-98.921875" y="-27" width="197.84375" height="54"></rect><g class="label" style="" transform="translate(-68.921875, -12)"><rect></rect><foreignobject width="137.84375" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
Typed row contexts
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-Engine-15" transform="translate(1488.0390625, 243)"><rect class="basic label-container" style="" x="-80.2421875" y="-27" width="160.484375" height="54"></rect><g class="label" style="" transform="translate(-50.2421875, -12)"><rect></rect><foreignobject width="100.484375" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
Pure classifier
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-Decision-17" transform="translate(1699.421875, 243)"><rect class="basic label-container" style="" x="-81.140625" y="-27" width="162.28125" height="54"></rect><g class="label" style="" transform="translate(-51.140625, -12)"><rect></rect><foreignobject width="102.28125" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
BatchDecision
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-AuditProj-19" transform="translate(1958.390625, 191)"><rect class="basic label-container" style="" x="-81.5859375" y="-27" width="163.171875" height="54"></rect><g class="label" style="" transform="translate(-51.5859375, -12)"><rect></rect><foreignobject width="103.171875" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
Audit projector
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-ReportProj-21" transform="translate(1958.390625, 295)"><rect class="basic label-container" style="" x="-127.828125" y="-27" width="255.65625" height="54"></rect><g class="label" style="" transform="translate(-97.828125, -12)"><rect></rect><foreignobject width="195.65625" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
Operational report projector
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-Audit-23" transform="translate(2252.046875, 191)"><rect class="basic label-container" style="" x="-96.9921875" y="-27" width="193.984375" height="54"></rect><g class="label" style="" transform="translate(-66.9921875, -12)"><rect></rect><foreignobject width="133.984375" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
AuditRecord DTOs
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-Report-25" transform="translate(2252.046875, 295)"><rect class="basic label-container" style="" x="-115.828125" y="-27" width="231.65625" height="54"></rect><g class="label" style="" transform="translate(-85.828125, -12)"><rect></rect><foreignobject width="171.65625" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
OperationalBatchReport
</p>
</span>
</div>
</foreignobject></g></g></g></g></g>
</svg>
</div>

The V2 work lives entirely inside the `Engine` box. Everything outside
that box is new in V3, and everything outside that box is where the
seven footguns try to make a comeback.

## Why this is the real test

The V2 lesson — *types make domain bugs structurally impossible* — is
true but incomplete. Anyone can write a clean pure function. The
question is whether you can wire it to a CSV parser, three repositories,
an audit log, and an ops dashboard without the CSV parser's strings, the
repository's nulls, the log's free-text status field, and the dashboard's
mutable counters creeping back into your decision types.

That is a different skill from "model the domain." It is the skill of
*holding a boundary*.

V3 is the test of that skill. The same four languages from V2 — C#, F#,
Haskell, Rust — get the same four boundary concerns added. The V2
scoring put Haskell first; the V3 scoring does not. The boundary
changes the answer.

The reason is intuitive once you look at it:

<div class="mermaid-diagram" style="width:100%">
<svg id="mermaid-figure-2" xmlns="http://www.w3.org/2000/svg" xlink="http://www.w3.org/1999/xlink" class="flowchart" viewbox="0 0 1767.19921875 450" role="graphics-document document" aria-roledescription="flowchart-v2" style="width:100%;height:auto;max-width:100%;display:block">
<style>#mermaid-figure-2{font-family:"trebuchet ms",verdana,arial,sans-serif;font-size:16px;fill:#000000;}@keyframes edge-animation-frame{from{stroke-dashoffset:0;}}@keyframes dash{to{stroke-dashoffset:0;}}#mermaid-figure-2 .edge-animation-slow{stroke-dasharray:9,5!important;stroke-dashoffset:900;animation:dash 50s linear infinite;stroke-linecap:round;}#mermaid-figure-2 .edge-animation-fast{stroke-dasharray:9,5!important;stroke-dashoffset:900;animation:dash 20s linear infinite;stroke-linecap:round;}#mermaid-figure-2 .error-icon{fill:#552222;}#mermaid-figure-2 .error-text{fill:#552222;stroke:#552222;}#mermaid-figure-2 .edge-thickness-normal{stroke-width:1px;}#mermaid-figure-2 .edge-thickness-thick{stroke-width:3.5px;}#mermaid-figure-2 .edge-pattern-solid{stroke-dasharray:0;}#mermaid-figure-2 .edge-thickness-invisible{stroke-width:0;fill:none;}#mermaid-figure-2 .edge-pattern-dashed{stroke-dasharray:3;}#mermaid-figure-2 .edge-pattern-dotted{stroke-dasharray:2;}#mermaid-figure-2 .marker{fill:#666;stroke:#666;}#mermaid-figure-2 .marker.cross{stroke:#666;}#mermaid-figure-2 svg{font-family:"trebuchet ms",verdana,arial,sans-serif;font-size:16px;}#mermaid-figure-2 p{margin:0;}#mermaid-figure-2 .label{font-family:"trebuchet ms",verdana,arial,sans-serif;color:#000000;}#mermaid-figure-2 .cluster-label text{fill:#333;}#mermaid-figure-2 .cluster-label span{color:#333;}#mermaid-figure-2 .cluster-label span p{background-color:transparent;}#mermaid-figure-2 .label text,#mermaid-figure-2 span{fill:#000000;color:#000000;}#mermaid-figure-2 .node rect,#mermaid-figure-2 .node circle,#mermaid-figure-2 .node ellipse,#mermaid-figure-2 .node polygon,#mermaid-figure-2 .node path{fill:#eee;stroke:#999;stroke-width:1px;}#mermaid-figure-2 .rough-node .label text,#mermaid-figure-2 .node .label text,#mermaid-figure-2 .image-shape .label,#mermaid-figure-2 .icon-shape .label{text-anchor:middle;}#mermaid-figure-2 .node .katex path{fill:#000;stroke:#000;stroke-width:1px;}#mermaid-figure-2 .rough-node .label,#mermaid-figure-2 .node .label,#mermaid-figure-2 .image-shape .label,#mermaid-figure-2 .icon-shape .label{text-align:center;}#mermaid-figure-2 .node.clickable{cursor:pointer;}#mermaid-figure-2 .root .anchor path{fill:#666!important;stroke-width:0;stroke:#666;}#mermaid-figure-2 .arrowheadPath{fill:#333333;}#mermaid-figure-2 .edgePath .path{stroke:#666;stroke-width:2.0px;}#mermaid-figure-2 .flowchart-link{stroke:#666;fill:none;}#mermaid-figure-2 .edgeLabel{background-color:white;text-align:center;}#mermaid-figure-2 .edgeLabel p{background-color:white;}#mermaid-figure-2 .edgeLabel rect{opacity:0.5;background-color:white;fill:white;}#mermaid-figure-2 .labelBkg{background-color:rgba(255, 255, 255, 0.5);}#mermaid-figure-2 .cluster rect{fill:hsl(0, 0%, 98.9215686275%);stroke:#707070;stroke-width:1px;}#mermaid-figure-2 .cluster text{fill:#333;}#mermaid-figure-2 .cluster span{color:#333;}#mermaid-figure-2 div.mermaidTooltip{position:absolute;text-align:center;max-width:200px;padding:2px;font-family:"trebuchet ms",verdana,arial,sans-serif;font-size:12px;background:hsl(-160, 0%, 93.3333333333%);border:1px solid #707070;border-radius:2px;pointer-events:none;z-index:100;}#mermaid-figure-2 .flowchartTitleText{text-anchor:middle;font-size:18px;fill:#000000;}#mermaid-figure-2 rect.text{fill:none;stroke-width:0;}#mermaid-figure-2 .icon-shape,#mermaid-figure-2 .image-shape{background-color:white;text-align:center;}#mermaid-figure-2 .icon-shape p,#mermaid-figure-2 .image-shape p{background-color:white;padding:2px;}#mermaid-figure-2 .icon-shape rect,#mermaid-figure-2 .image-shape rect{opacity:0.5;background-color:white;fill:white;}#mermaid-figure-2 .label-icon{display:inline-block;height:1em;overflow:visible;vertical-align:-0.125em;}#mermaid-figure-2 .node .label-icon path{fill:currentColor;stroke:revert;stroke-width:revert;}#mermaid-figure-2 :root{--mermaid-font-family:"trebuchet ms",verdana,arial,sans-serif;}</style>
<g><marker id="mermaid-1779381583559_flowchart-v2-pointEnd" class="marker flowchart-v2" viewbox="0 0 10 10" refx="5" refy="5" markerunits="userSpaceOnUse" markerwidth="8" markerheight="8" orient="auto"><path d="M 0 0 L 10 5 L 0 10 z" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></path></marker><marker id="mermaid-1779381583559_flowchart-v2-pointStart" class="marker flowchart-v2" viewbox="0 0 10 10" refx="4.5" refy="5" markerunits="userSpaceOnUse" markerwidth="8" markerheight="8" orient="auto"><path d="M 0 5 L 10 10 L 10 0 z" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></path></marker><marker id="mermaid-1779381583559_flowchart-v2-circleEnd" class="marker flowchart-v2" viewbox="0 0 10 10" refx="11" refy="5" markerunits="userSpaceOnUse" markerwidth="11" markerheight="11" orient="auto"><circle cx="5" cy="5" r="5" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></circle></marker><marker id="mermaid-1779381583559_flowchart-v2-circleStart" class="marker flowchart-v2" viewbox="0 0 10 10" refx="-1" refy="5" markerunits="userSpaceOnUse" markerwidth="11" markerheight="11" orient="auto"><circle cx="5" cy="5" r="5" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></circle></marker><marker id="mermaid-1779381583559_flowchart-v2-crossEnd" class="marker cross flowchart-v2" viewbox="0 0 11 11" refx="12" refy="5.2" markerunits="userSpaceOnUse" markerwidth="11" markerheight="11" orient="auto"><path d="M 1,1 l 9,9 M 10,1 l -9,9" class="arrowMarkerPath" style="stroke-width: 2; stroke-dasharray: 1, 0;"></path></marker><marker id="mermaid-1779381583559_flowchart-v2-crossStart" class="marker cross flowchart-v2" viewbox="0 0 11 11" refx="-1" refy="5.2" markerunits="userSpaceOnUse" markerwidth="11" markerheight="11" orient="auto"><path d="M 1,1 l 9,9 M 10,1 l -9,9" class="arrowMarkerPath" style="stroke-width: 2; stroke-dasharray: 1, 0;"></path></marker><g class="root"><g class="clusters"><g class="cluster" id="Inside" data-look="classic"><rect style="" x="8" y="161" width="966.84375" height="128"></rect><g class="cluster-label" transform="translate(438.0546875, 161)"><foreignobject width="106.734375" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
The typed core
</p>
</span>
</div>
</foreignobject></g></g><g class="cluster" id="Outside" data-look="classic"><rect style="" x="994.84375" y="8" width="764.35546875" height="434"></rect><g class="cluster-label" transform="translate(1316.560546875, 8)"><foreignobject width="120.921875" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
The messy world
</p>
</span>
</div>
</foreignobject></g></g></g><g class="edgePaths"><path d="M1159.946,87L1149.814,93.167C1139.681,99.333,1119.417,111.667,1109.285,124C1099.152,136.333,1099.152,148.667,986.218,164.238C873.284,179.81,647.417,198.62,534.483,208.025L421.549,217.43" id="L_CSVcells_Row_0" class="edge-thickness-normal edge-pattern-dotted edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_CSVcells_Row_0" data-points="W3sieCI6MTE1OS45NDU4MDA3ODEyNSwieSI6ODd9LHsieCI6MTA5OS4xNTIzNDM3NSwieSI6MTI0fSx7IngiOjEwOTkuMTUyMzQzNzUsInkiOjE2MX0seyJ4Ijo0MTcuNTYyNSwieSI6MjE3Ljc2MTkxMDU3MDk2NTN9XQ==" marker-end="url(#mermaid-1779381583559_flowchart-v2-pointEnd)"></path><path d="M1415.57,87L1395.233,93.167C1374.895,99.333,1334.221,111.667,1313.884,124C1293.547,136.333,1293.547,148.667,1199.88,163.447C1106.213,178.226,918.879,195.453,825.213,204.066L731.546,212.679" id="L_RepoCalls_Lookups_0" class="edge-thickness-normal edge-pattern-dotted edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_RepoCalls_Lookups_0" data-points="W3sieCI6MTQxNS41Njk2NDExMTMyODEyLCJ5Ijo4N30seyJ4IjoxMjkzLjU0Njg3NSwieSI6MTI0fSx7IngiOjEyOTMuNTQ2ODc1LCJ5IjoxNjF9LHsieCI6NzI3LjU2MjUsInkiOjIxMy4wNDU3MDg2NDEwODg0fV0=" marker-end="url(#mermaid-1779381583559_flowchart-v2-pointEnd)"></path><path d="M777.563,237.503L721.861,246.086C666.159,254.668,554.755,271.834,624.982,286.584C695.208,301.333,947.065,313.667,1078.98,325.54C1210.895,337.413,1222.869,348.827,1228.855,354.533L1234.842,360.24" id="L_Decision_AuditText_0" class="edge-thickness-normal edge-pattern-dotted edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_Decision_AuditText_0" data-points="W3sieCI6Nzc3LjU2MjUsInkiOjIzNy41MDI2NjE1MjU0Mzk2Nn0seyJ4Ijo0NDMuMzUxNTYyNSwieSI6Mjg5fSx7IngiOjExOTguOTIxODc1LCJ5IjozMjZ9LHsieCI6MTIzNy43Mzc1NDg4MjgxMjUsInkiOjM2M31d" marker-end="url(#mermaid-1779381583559_flowchart-v2-pointEnd)"></path><path d="M883.202,252L888.797,258.167C894.392,264.333,905.583,276.667,1017.518,289C1129.453,301.333,1342.133,313.667,1448.473,325.333C1554.813,337,1554.813,348,1554.813,353.5L1554.813,359" id="L_Decision_ReportPayload_0" class="edge-thickness-normal edge-pattern-dotted edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_Decision_ReportPayload_0" data-points="W3sieCI6ODgzLjIwMTUzODA4NTkzNzUsInkiOjI1Mn0seyJ4Ijo5MTYuNzczNDM3NSwieSI6Mjg5fSx7IngiOjE1NTQuODEyNSwieSI6MzI2fSx7IngiOjE1NTQuODEyNSwieSI6MzYzfV0=" marker-end="url(#mermaid-1779381583559_flowchart-v2-pointEnd)"></path><path d="M1293.352,87L1313.689,93.167C1334.026,99.333,1374.701,111.667,1395.038,124C1415.375,136.333,1415.375,148.667,1336.782,163.869C1258.189,179.071,1101.003,197.143,1022.41,206.179L943.818,215.214" id="L_CSVcells_Decision_0" class="edge-thickness-normal edge-pattern-dotted edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_CSVcells_Decision_0" data-points="W3sieCI6MTI5My4zNTIyMzM4ODY3MTg4LCJ5Ijo4N30seyJ4IjoxNDE1LjM3NSwieSI6MTI0fSx7IngiOjE0MTUuMzc1LCJ5IjoxNjF9LHsieCI6OTM5Ljg0Mzc1LCJ5IjoyMTUuNjcxMzQ0NzY2NjA5Nn1d" marker-end="url(#mermaid-1779381583559_flowchart-v2-pointEnd)"></path><path d="M1560.792,87L1573.623,93.167C1586.454,99.333,1612.116,111.667,1624.946,124C1637.777,136.333,1637.777,148.667,1522.12,164.334C1406.462,180.002,1175.146,199.005,1059.488,208.506L943.83,218.007" id="L_RepoCalls_Decision_0" class="edge-thickness-normal edge-pattern-dotted edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_RepoCalls_Decision_0" data-points="W3sieCI6MTU2MC43OTE4NzAxMTcxODc1LCJ5Ijo4N30seyJ4IjoxNjM3Ljc3NzM0Mzc1LCJ5IjoxMjR9LHsieCI6MTYzNy43NzczNDM3NSwieSI6MTYxfSx7IngiOjkzOS44NDM3NSwieSI6MjE4LjMzNDM5NjI5MzY3Nzg4fV0=" marker-end="url(#mermaid-1779381583559_flowchart-v2-pointEnd)"></path><path d="M1298.646,363L1306.088,356.833C1313.529,350.667,1328.413,338.333,1255.089,326C1181.766,313.667,1020.234,301.333,939.469,289.667C858.703,278,858.703,267,858.703,261.5L858.703,256" id="L_AuditText_Decision_0" class="edge-thickness-normal edge-pattern-dotted edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_AuditText_Decision_0" data-points="W3sieCI6MTI5OC42NDU3NTE5NTMxMjUsInkiOjM2M30seyJ4IjoxMzQzLjI5Njg3NSwieSI6MzI2fSx7IngiOjg1OC43MDMxMjUsInkiOjI4OX0seyJ4Ijo4NTguNzAzMTI1LCJ5IjoyNTJ9XQ==" marker-end="url(#mermaid-1779381583559_flowchart-v2-pointEnd)"></path></g><g class="edgeLabels"><g class="edgeLabel" transform="translate(1099.15234375, 124)"><g class="label" data-id="L_CSVcells_Row_0" transform="translate(-28.90625, -12)"><foreignobject width="57.8125" height="24">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel">
<p>
parse to
</p>
</span>
</div>
</foreignobject></g></g><g class="edgeLabel" transform="translate(1293.546875, 124)"><g class="label" data-id="L_RepoCalls_Lookups_0" transform="translate(-39.578125, -12)"><foreignobject width="79.15625" height="24">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel">
<p>
translate to
</p>
</span>
</div>
</foreignobject></g></g><g class="edgeLabel" transform="translate(1198.921875, 326)"><g class="label" data-id="L_Decision_AuditText_0" transform="translate(-32.90625, -12)"><foreignobject width="65.8125" height="24">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel">
<p>
project to
</p>
</span>
</div>
</foreignobject></g></g><g class="edgeLabel" transform="translate(1554.8125, 326)"><g class="label" data-id="L_Decision_ReportPayload_0" transform="translate(-32.90625, -12)"><foreignobject width="65.8125" height="24">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel">
<p>
project to
</p>
</span>
</div>
</foreignobject></g></g><g class="edgeLabel" transform="translate(1415.375, 124)"><g class="label" data-id="L_CSVcells_Decision_0" transform="translate(-62.25, -12)"><foreignobject width="124.5" height="24">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel">
<p>
must not flow into
</p>
</span>
</div>
</foreignobject></g></g><g class="edgeLabel" transform="translate(1637.77734375, 124)"><g class="label" data-id="L_RepoCalls_Decision_0" transform="translate(-62.25, -12)"><foreignobject width="124.5" height="24">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel">
<p>
must not flow into
</p>
</span>
</div>
</foreignobject></g></g><g class="edgeLabel" transform="translate(1343.296875, 326)"><g class="label" data-id="L_AuditText_Decision_0" transform="translate(-81.375, -12)"><foreignobject width="162.75" height="24">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel">
<p>
must not flow back into
</p>
</span>
</div>
</foreignobject></g></g></g><g class="nodes"><g class="node default" id="flowchart-CSVcells-0" transform="translate(1204.30859375, 60)"><rect class="basic label-container" style="" x="-124.7109375" y="-27" width="249.421875" height="54"></rect><g class="label" style="" transform="translate(-94.7109375, -12)"><rect></rect><foreignobject width="189.421875" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
CSV cells: strings, nullable
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-RepoCalls-1" transform="translate(1504.61328125, 60)"><rect class="basic label-container" style="" x="-125.59375" y="-27" width="251.1875" height="54"></rect><g class="label" style="" transform="translate(-95.59375, -12)"><rect></rect><foreignobject width="191.1875" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
Repository calls: IO, failure
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-AuditText-2" transform="translate(1266.0625, 390)"><rect class="basic label-container" style="" x="-109.15625" y="-27" width="218.3125" height="54"></rect><g class="label" style="" transform="translate(-79.15625, -12)"><rect></rect><foreignobject width="158.3125" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
Audit log: external text
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-ReportPayload-3" transform="translate(1554.8125, 390)"><rect class="basic label-container" style="" x="-129.59375" y="-27" width="259.1875" height="54"></rect><g class="label" style="" transform="translate(-99.59375, -12)"><rect></rect><foreignobject width="199.1875" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
Ops report: serialised JSON
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-Mode-4" transform="translate(118.3671875, 225)"><rect class="basic label-container" style="" x="-75.3671875" y="-27" width="150.734375" height="54"></rect><g class="label" style="" transform="translate(-45.3671875, -12)"><rect></rect><foreignobject width="90.734375" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
UploadMode
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-Row-5" transform="translate(330.6484375, 225)"><rect class="basic label-container" style="" x="-86.9140625" y="-27" width="173.828125" height="54"></rect><g class="label" style="" transform="translate(-56.9140625, -12)"><rect></rect><foreignobject width="113.828125" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
ParsedFuelRow
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-Lookups-6" transform="translate(597.5625, 225)"><rect class="basic label-container" style="" x="-130" y="-39" width="260" height="78"></rect><g class="label" style="" transform="translate(-100, -24)"><rect></rect><foreignobject width="200" height="48">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table; white-space: break-spaces; line-height: 1.5; max-width: 200px; text-align: center; width: 200px;">
<span class="nodeLabel">
<p>
VehicleLookupResult, DuplicateCheckResult
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-Decision-7" transform="translate(858.703125, 225)"><rect class="basic label-container" style="" x="-81.140625" y="-27" width="162.28125" height="54"></rect><g class="label" style="" transform="translate(-51.140625, -12)"><rect></rect><foreignobject width="102.28125" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
BatchDecision
</p>
</span>
</div>
</foreignobject></g></g></g></g></g>
</svg>
</div>

The arrows that *should* be there are typed translations at the edge.
The arrows that *should not* be there are the bugs. A junior writing
this code for the first time has to actively resist drawing them. A
mid-level engineer writes them by accident in the third file of a six-file
PR. The question V3 asks is: **which language makes the missing arrows
hard to draw?**

## The five risks at every boundary

When you sit a junior down in front of the V3 pipeline, here are the
five concrete things they tend to ship. They are not exotic. Each one
is a single line of code at the edge that quietly undoes the V2 lesson.

1. **Raw string statuses sneaking back into domain code.** The audit
   layer needs an external status text like `"accepted_with_warnings"`.
   Once that string is in scope, it is one paste away from being the
   *input* to the next layer's switch statement. Strong tip: produce
   the string at the *very last* boundary, never before.

2. **Nullable/default DTO construction violating non-empty invariants.**
   In V2 we worked hard to make `AcceptedTransaction` carry a real
   `Vehicle` and a real `TransactionKey`. The DTO layer hands you back
   nullable strings, and `with { TransactionKey = null }` compiles.
   F#'s `[<CLIMutable>]` records, C#'s record `with` updates, Rust's
   `Option<String>` — every one of these can hand you an "accepted"
   row whose key is empty if you are not paying attention.

3. **Loose numeric types for money and quantity.** The pure engine
   uses `decimal`/`Rational`/`Decimal` because rounding matters. The
   DTO layer takes a string and parses it. If you parse to `f64` /
   `Double` because that is what `parse::<f64>()` returns, you have
   reintroduced floating point into a financial calculation at the
   boundary. The V3 audit caught this in Rust: monetary values
   round-tripped through `f64`.

4. **Repository failures losing detail.** When the vehicle-lookup
   service times out, the typed-error part of you wants to say
   `RepositoryError::TimedOut { duration: 7s, endpoint: "..." }`. The
   tired part of you collapses that into "this row is fatal" and
   forgets the detail. The V3 audit caught this in both C# and Rust:
   the repository result was translated *first* into a DTO status
   string, *then* re-parsed back into a domain `Fatal` variant. The
   round-trip discarded everything in between.

5. **Summary/report logic duplicating per-row decision logic.** The
   ops report needs to count "accepted with warnings." The naive
   implementation iterates the raw input rows and re-decides what
   counts as accepted. Now you have two classifiers, and they will
   drift. The V2 lesson — *derive the summary from decisions* — has to
   hold at the report boundary too. The V3 phase briefing called this
   out specifically: the counterexample is the report projector that
   inspects `csvRows` and tracks `accepted` and `skipped` in mutable
   counters; the good shape is the projector that reads `BatchDecision`
   and never looks at the raw input again.

All four V3 implementations close some of these risks; none closes all
five. The honest cross-language picture is the next chapter.

## What V2 was and was not

It is worth being explicit about what the V2 audit *did* test, so the
V3 lesson lands without sounding like a contradiction.

V2 took the pure decision engine and pushed five rounds of change
through it: split a monolith into modules, add a new `Quarantined`
outcome with typed reasons, split `Recovery` mode into conservative and
aggressive variants, add a thin DTO mapping layer, score the result.
At every step the V2 audit checked the same V1 footguns — raw status
strings, nullable accepted transactions, boolean soup in duplicate
state, mutated summaries — and confirmed that each idiomatic
implementation either prevented the bug or made it harder to write.

What V2 did *not* push hard on:

- a CSV-shaped import that has to handle bad cells, not just
  well-typed DTOs;
- repository lookups that can fail mid-batch and have to be translated
  into typed outcomes;
- a separate audit projection that emits records to a log other systems
  consume;
- an ops report that consumers will read once and assume is honest.

Each of those is a new boundary. Each one re-opens the same V1
footguns. V3 made that explicit by adding all four at once.

The four V3 implementations did not regress on V2 rules. The decision
engine in each language is still pure, still produces typed outcomes,
still derives the summary from decisions. What V3 surfaced is the
*surface area* the integration adds. That surface area is where the
ranking flipped.

## What's next

[Chapter 12](12-boundary-side-by-side.ipynb) walks all four engines
through all four boundary concerns side-by-side, with real code from
each implementation. It is the longest chapter in the book because the
boundary is where most of the engineering effort lives in production.

[Chapter 13](13-final-takeaway.ipynb) is the final takeaway: what to
recommend a junior developer do in a real .NET shop, given everything
the two halves of this book have shown.

If you want the scoring rationale up front, the [V3 results
document](https://github.com/tedk99/experiment-normalcsharp/blob/main/docs/v3-results.md)
has every score and verdict. The short version: the V2 winner is not the
V3 winner, and the reason is the boundary.